# Part 3 - Multi-Agent System with Microsoft AutoGen

*Last updated: November 2026*

You will build a customer onboarding flow with AutoGen (version 0.2 API):
1. A single **ConversableAgent** chatbot.
2. Specialised agents - registration, sales and help desk - plus a **UserProxyAgent** that relays *your* replies.
3. A **sequence of chats** (`initiate_chats`) where each step is summarised into **structured JSON**
   with `summary_method="reflection_with_llm"` and passed on to the next step.

The models run on **Groq** through its OpenAI-compatible API (Groq has a free tier).

## How to run this notebook

1. Set up the course environment once, in the course folder: `uv sync`.
   New to coding? Follow `BEGINNER_SETUP.md` step by step. Otherwise see `LOCAL_SETUP.MD`.
2. Select the **`.venv`** Python kernel (*Python 3.12*) for this notebook.
3. Copy `.env.example` (course folder) to `.env` and add your keys. The notebook finds `.env` automatically.

**Keys and settings used in this notebook**
- `GROQ_API_KEY` (required)
- `GROQ_MODEL` (optional)

Run the cells from top to bottom.

> **Compatibility update**
> - The recording used `llama-3.3-70b-versatile`, which Groq no longer offers. The default is now **`openai/gpt-oss-120b`**.
>   You can change the model without editing code: set `GROQ_MODEL` in `.env` to a current model
>   from <https://console.groq.com/docs/models>.
> - Each summary step adds `"summary_role": "user"`, so the summary request is sent as a user message. Some Groq models
>   return empty summaries when the request arrives as a final system message.
> - `"price": [0, 0]` tells AutoGen how to count cost for Groq models and removes a "model not found" cost warning.
> - No OpenAI key is needed for this notebook.

In [ ]:
# Setup check: run this cell first.
# Nothing is installed here. All packages were installed once with `uv sync` (see LOCAL_SETUP.MD).
import sys
from pathlib import Path

python_version = sys.version.split()[0]

if sys.version_info[:2] != (3, 12):
    print(f"Warning: this course is tested on Python 3.12, but this kernel runs Python {python_version}.")
    print("Select the .venv kernel - see 'How to run this notebook' above.")
elif Path(sys.prefix).name != ".venv":
    print("Warning: this kernel is not the course .venv environment, so some packages may be missing.")
    print("Select the .venv kernel - see 'How to run this notebook' above.")
else:
    print("Ready: course .venv kernel | Python", python_version)

In [ ]:
# Load API keys from the .env file, so keys are never written in the notebook.
# load_dotenv() looks for .env in this folder and then in each parent folder (the course folder).
import os

from dotenv import load_dotenv

load_dotenv()

REQUIRED_KEYS = ['GROQ_API_KEY']

missing = [key for key in REQUIRED_KEYS if not os.getenv(key)]
if missing:
    print("Missing required keys:", ", ".join(missing), "- see 'How to run this notebook' above.")
else:
    print("All required API keys are loaded.")

In [ ]:
# Hide library deprecation warnings so the lesson output stays readable
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import pprint

# Configure the model (Groq, OpenAI-compatible endpoint)
GROQ_MODEL = os.getenv("GROQ_MODEL") or "openai/gpt-oss-120b"

config_list = [{
    "model": GROQ_MODEL,
    "api_key": os.environ.get("GROQ_API_KEY"),
    "base_url": "https://api.groq.com/openai/v1",
    "price": [0, 0],
}]
print("Using Groq model:", GROQ_MODEL)

In [ ]:
from autogen import ConversableAgent, AssistantAgent, UserProxyAgent
from autogen import initiate_chats

agent = ConversableAgent(
    name="chatbot",
    llm_config={"config_list": config_list},
    human_input_mode="NEVER",
)

In [ ]:
reply = agent.generate_reply(
    messages=[{"content": "Tell me a joke.", "role": "user"}]
)
print(reply)

### The three agent types used below
1. **ConversableAgent** - holds a structured conversation, such as onboarding a new customer.
2. **AssistantAgent** - focuses on getting a task done, often with expertise in one area.
3. **UserProxyAgent** - stands in for the user and passes the user's replies to the other agents.

In [ ]:
# Define the RegistrationAgent for collecting user details
registration_agent = ConversableAgent(
    name="User Registration Agent",
    system_message='''You are a friendly registration assistant.
    Your task is to collect the user's full name, company name, and the product they are interested in.
    Do not ask for additional details. Respond with 'END_SESSION' once all details are obtained.''',
    llm_config={"config_list": config_list},
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [ ]:
# Define the HelpDeskAgent for troubleshooting
help_desk_agent = AssistantAgent(
    name="Help Desk Agent",
    system_message='''You are a knowledgeable support assistant.
    Your role is to assist users in buying our product.
    Offer clear steps, share useful resources, and escalate if required.
    Respond with 'END_SESSION' once the issue is addressed.''',
    llm_config={"config_list": config_list},
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [ ]:
# Define the SalesAgent for product recommendations
sales_agent = AssistantAgent(
    name="Sales Recommendation Agent",
    system_message='''You are a sales assistant.
    Based on user information, suggest the most suitable product and provide additional details.
    If the user expresses interest, proceed with gathering purchase intent details.
    Respond with 'END_SESSION' when user decides to proceed or not.''',
    llm_config={"config_list": config_list},
    code_execution_config=False,
    human_input_mode="NEVER",
)

In [ ]:
# Define the RelayAgent for communication handling (you type its replies)
relay_agent = UserProxyAgent(
    name="Interaction Relay Agent",
    system_message='''You act as a bridge between users and specialized agents.
    You do not process requests yourself but ensure smooth communication.''',
    llm_config=False,
    code_execution_config=False,
    human_input_mode="ALWAYS",
)

### Run the onboarding conversation

The next cell asks **you** to reply as the user. Type your answers (for example a name, a company and a
product) into the input box and press Enter. Press Enter on an empty box to let the agent reply
automatically, or type `exit` to end the current step.

In [ ]:
conversation_flow = [
    {
        "sender": registration_agent,
        "recipient": relay_agent,
        "message": "Hi! I'm here to help with your onboarding. Please provide your full name, company, and the product you're interested in.",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Extract user details as JSON: {'full_name': '', 'company': '', 'product': ''}",
            "summary_role": "user",
        },
        "max_turns": 2,
        "clear_history": True
    },
    {
        "sender": relay_agent,
        "recipient": sales_agent,
        "message": "Based on user details, suggest a suitable product.",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Summarize recommended products as JSON: {'recommended_product': '', 'justification': ''}",
            "summary_role": "user",
        },
        "max_turns": 3,
        "clear_history": False
    },
    {
        "sender": relay_agent,
        "recipient": help_desk_agent,
        "message": "Based on sale recommendation, closen the conversation with formal sale pitch",
        "summary_method": "reflection_with_llm",
        "summary_args": {
            "summary_prompt": "Summarize the assistance provided as JSON: {'resolution_steps': []}",
            "summary_role": "user",
        },
        "max_turns": 3,
        "clear_history": False
    }
]

# Execute the conversations
session_results = initiate_chats(conversation_flow)
pprint.pp(session_results)

### What each key in `conversation_flow` means

- **sender**: The agent that sends the message.
- **recipient**: The agent that receives it.
- **message**: The text of the message.
- **summary_method**: How to summarize the conversation, often with an LLM.
- **summary_args**: Parameters for the summary method (`summary_prompt` asks for JSON; `summary_role` sets the role of that request).
- **max_turns**: The maximum number of exchanges allowed in this part of the conversation.
- **clear_history**: Whether to clear the conversation history after this chat.

In [ ]:
# The structured JSON summary produced by each step
for step, chat_result in enumerate(session_results, start=1):
    print(f"Step {step} summary:", chat_result.summary)